# Figure 4 — Thy1 mixture discriminability

**Population:** Thy1 10× only. **Claim:** anesthesia reduces reliable spatial and/or temporal separation between similar mixtures. TH and DAT are excluded.

## 1. Repository bootstrap and explicit inputs

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from tqdm.auto import tqdm
HERE = Path.cwd().resolve()
REPO = next((p for p in (HERE, *HERE.parents) if (p / 'analysis').is_dir()), None)
if REPO is None: raise RuntimeError('Could not locate the ODyn-analysis repository root.')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
from analysis.figures.paths import imaging_root, repo_path
MANIFEST = repo_path('analysis', 'stage0', 'ketxyl_16odor_session_manifest.csv')
IMAGING_ROOT = imaging_root()  # set ODYN_IMAGING_ROOT for this computer/server
OUTPUT = repo_path('analysis', 'figures', 'figure4', 'outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
MIN_GEOMETRY_TRIALS = 3  # per odor, state, and session
print('Repository:', REPO)
print('Output:', OUTPUT)

## 2. Inventory Thy1 inputs

In [ ]:
from analysis.figures.session_data import available_sessions
inventory = pd.DataFrame(available_sessions(MANIFEST, IMAGING_ROOT, objective='10x'))
inventory['line'] = inventory.population.str.split('-').str[0]
inventory = inventory[inventory.line == 'Thy1'].copy()
inventory.to_csv(OUTPUT / 'thy1_10x_session_inventory.csv', index=False)
display(inventory[['group_id', 'mouse', 'available', 'grouped_path']])

## 3. Calculate integrated, spatiotemporal, cosine, and cumulative geometry

Crossnobis uses trial-level glomerular population vectors. Integrated geometry averages 0–4 s; spatiotemporal geometry concatenates four 1-second vectors; cosine is a gain-insensitive, non-crossvalidated secondary description.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import analyze_session
rows = []
geometry_inputs = inventory.loc[inventory.available].to_dict('records')
for row in tqdm(geometry_inputs, desc='Thy1 mixture geometry', unit='session'):
    rows.extend(analyze_session(row, Path(row['grouped_path'])))
geometry = pd.DataFrame(rows)
geometry.to_csv(OUTPUT / 'thy1_mixture_geometry_long.csv', index=False)
summary = geometry.drop_duplicates(['group_id', 'mouse', 'line', 'state', 'pair'])
summary.to_csv(OUTPUT / 'thy1_mixture_geometry_session_summary.csv', index=False)
summary['min_trials'] = summary[['n_a', 'n_b']].min(axis=1)
excluded = summary[summary.min_trials < MIN_GEOMETRY_TRIALS].copy()
primary = summary[summary.min_trials >= MIN_GEOMETRY_TRIALS].copy()
valid_keys = primary[['group_id', 'state', 'pair']].drop_duplicates()
primary_long = geometry.merge(valid_keys, on=['group_id', 'state', 'pair'], how='inner')
excluded.to_csv(OUTPUT / 'thy1_geometry_low_trial_exclusions.csv', index=False)
print(f'Primary comparisons: {len(primary)}; low-trial comparisons excluded: {len(excluded)}')
display(excluded[['group_id', 'mouse', 'state', 'pair', 'n_a', 'n_b']])

## 4. Panel 4A — paired awake/anesthetized discriminability

Thin lines are sessions. The thick trajectory averages sessions within mouse before taking the across-mouse median.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import plot_state_comparison
from IPython.display import Image, display
panel = OUTPUT / 'panel_4A_thy1_mixture_geometry.png'
plot_state_comparison(panel, primary)
display(Image(filename=str(panel)))
print('Saved:', panel)

## 5. Panel 4B — when mixture information emerges

Cumulative distance asks how much reliable information is available after the first, second, third, and fourth seconds.

In [ ]:
from analysis.figures.figure2.make_10x_mixture_geometry import plot_cumulative
panel = OUTPUT / 'panel_4B_thy1_cumulative_geometry.png'
plot_cumulative(panel, primary_long)
display(Image(filename=str(panel)))
print('Saved:', panel)

## 6. Interpretation guardrails

A crossnobis decrease can reflect response compression and/or reduced pattern reliability. A simultaneous cosine decrease supports convergence of map direction after removing uniform gain. Neither establishes that DA lateral inhibition caused the Thy1 change because populations were recorded in separate animals.